# Stage 5 trial: match OIE entities to schema.org types

First experiment for the schema-canonicalization stage. Goal: take the free-form types the OIE step assigned to an article's entities and, for each **distinct type**, retrieve the nearest schema.org classes by embedding similarity. Every entity then inherits the match found for its type.

**Reusable across articles.** Set `ARTICLE_DIR` in the first code cell to the folder you want to work on (e.g. `politics_article`, `world_cup_article`). Inputs are read from there and all outputs are written back into it. Shared schema.org assets live in `schema_org_files/` and are the same for every article.

```
dataset_creation/
├── schema_matching_trial.ipynb      <- this notebook
├── schema_org_files/                <- shared: schemaorg dump, parsed + enriched classes, builder
└── <article>/                       <- per-article: entities file + results outputs
```

Pipeline in this notebook:
1. Parse `schemaorg-current-https.jsonld` into a flat list of classes, each with a **hierarchy path** built from `rdfs:subClassOf`.
2. Build an embedding string per class: `Label (ancestor > ancestor): comment`.
3. Load the article's entities and build one query per **distinct OIE type** (`type: definition`, mention excluded).
4. Embed both pools with a sentence-transformer and do top-k cosine retrieval per type, then map each type's match onto its entities.
5. Inspect results, then compare against the LLM-enriched schema definitions.

This is only the **retrieval** half of Stage 5 — the LLM multiple-choice verifier is a later step.

In [ ]:
# Run this once to install dependencies into the active kernel.
#   - sentence-transformers: the embedding model loader (pulls in torch, a large first download)
#   - numpy: array math for the similarity step
# After it finishes the first time, you can skip this cell on later runs.
%pip install -q sentence-transformers numpy

In [1]:
import json, re
from pathlib import Path

# --- Which article to work on -------------------------------------------------------------
# Point this at any per-article folder (e.g. "politics_article", "world_cup_article").
# That folder must contain the entities file named below; outputs are written back into it.
ARTICLE_DIR = Path("politics_article")
ENTITIES_FILE = "OIE_entities_claude.json"   # {"entities": [...], "types": [...]} ; "types" optional

# --- Shared schema.org assets (same for every article) -----------------------------------
SCHEMA_DIR = Path("schema_org_files")
SCHEMA_JSONLD = SCHEMA_DIR / "schemaorg-current-https.jsonld"  # the full schema.org vocabulary dump

# --- Derived input/output paths ----------------------------------------------------------
OIE_ENTITIES = ARTICLE_DIR / ENTITIES_FILE
print(f"article: {ARTICLE_DIR}  |  entities: {OIE_ENTITIES}")

# --- Knobs ---
# Matching is done at the OIE *type* level: each DISTINCT free-form type (label + definition) is
# canonicalized to schema.org once, and every entity of that type inherits the result. The
# mention is deliberately NOT embedded, so the same type can't reshuffle per mention; a mention
# typed under several types (e.g. Mexico as Country AND National football team) still gets one
# schema match per typing.
TOP_K = 5                                # how many candidate schema classes to keep per OIE type
MODEL_NAME = "intfloat/e5-base-v2"    # the embedding model.
                                         # swappable: BAAI/bge-m3 (stronger, ~2GB),
                                         #            thenlper/gte-base, intfloat/e5-base-v2

# BGE-family models are trained so that *queries* get a short instruction prefix but the
# *documents* (here: the schema class texts) do not. We prepend this to each type query only.
# e.g. query becomes: "Represent this sentence for searching relevant passages: Country: A sovereign..."
QUERY_INSTRUCTION = "Represent this sentence for searching relevant passages: "

article: politics_article  |  entities: politics_article/OIE_entities_claude.json


## 1. Parse schema.org into classes with hierarchy paths

In [2]:
def plain(value) -> str:
    """Normalize a schema.org field into a clean plain string.

    Two quirks in the dump we handle here:
    1. A field can be a language-tagged dict instead of a string, e.g.
         {'@language': 'en', '@value': 'An intangible type...'}  ->  'An intangible type...'
    2. Comments often contain HTML, e.g.
         'A <a href="...">city</a> or town.'  ->  'A city or town.'
    """
    if isinstance(value, dict):
        value = value.get("@value", "")
    value = re.sub(r"<[^>]+>", " ", value or "")   # drop HTML tags
    return re.sub(r"\s+", " ", value).strip()      # collapse whitespace

def as_list(x):
    """Wrap a single value in a list so we can loop uniformly.
    In the dump a field is sometimes one dict and sometimes a list of dicts, e.g.
      rdfs:subClassOf: {'@id': 'schema:Place'}           -> [{'@id': 'schema:Place'}]
      rdfs:subClassOf: [{'@id': 'A'}, {'@id': 'B'}]      -> unchanged
    """
    if x is None:
        return []
    return x if isinstance(x, list) else [x]

# The dump is one big JSON-LD object; every type/property/enum lives in the "@graph" list.
raw = json.loads(SCHEMA_JSONLD.read_text())
graph = raw["@graph"]

# Walk the graph and keep only the actual classes (types) defined by schema.org itself.
classes = {}  # maps 'schema:Country' -> {id, label, comment, parents}
for node in graph:
    # A node is a class iff its @type includes "rdfs:Class".
    # (The graph also holds rdf:Property nodes, enumeration members, etc. — skip those.)
    types = as_list(node.get("@type"))
    if "rdfs:Class" not in types:
        continue

    cid = node["@id"]                       # e.g. 'schema:Country'
    # schema.org imports external vocabularies (gs1:, dcterms:, ...). Keep only native schema: types.
    if not cid.startswith("schema:"):
        continue

    # Direct parent classes (the "is-a" links). A class can have several parents; we keep
    # only schema: parents and drop cross-vocabulary ones like 'cmns-ge:GeopoliticalEntity'.
    # e.g. Country -> ['schema:AdministrativeArea']
    parents = [p["@id"] for p in as_list(node.get("rdfs:subClassOf"))
               if isinstance(p, dict) and p.get("@id", "").startswith("schema:")]

    classes[cid] = {
        "id": cid,
        "label": plain(node.get("rdfs:label")) or cid.split(":")[-1],  # 'Country'
        "comment": plain(node.get("rdfs:comment")),                    # 'A country.'  (the definition)
        "parents": parents,                                            # ['schema:AdministrativeArea']
    }

print(f"{len(classes)} schema.org classes parsed")

933 schema.org classes parsed


In [3]:
def hierarchy_path(cid, _seen=None):
    """Walk the 'is-a' chain up to the root and return the ancestor labels, root-first.

    Example: hierarchy_path('schema:Country') -> ['Thing', 'Place', 'AdministrativeArea']
    (the class itself, 'Country', is NOT included — only its ancestors).

    Notes:
    - A class can have multiple parents; we follow only the FIRST one (parents[0]) to get a
      single readable path. Good enough for this trial.
    - _seen guards against cycles so the recursion can't loop forever.
    """
    _seen = _seen or set()
    if cid in _seen or cid not in classes:
        return []
    _seen.add(cid)
    parents = classes[cid]["parents"]
    if not parents:                       # reached the root (Thing has no schema: parent)
        return []
    p = parents[0]
    # recurse up to the root first, then append this parent's label at the end (root-first order)
    return hierarchy_path(p, _seen) + [classes[p]["label"]] if p in classes else []

def class_embedding_text(c):
    """Build the single string we feed to the embedding model for a class.

    Format: "Label (ancestor > ancestor > ...): comment"
    Example (Country):
        "Country (Thing > Place > AdministrativeArea): A country."

    Why include the hierarchy path? It gives the embedding extra context about WHERE the class
    sits, so e.g. 'Country' (a Place) and a hypothetical 'Country' (a music genre) wouldn't
    collide. The comment carries the meaning; the path carries the position.
    """
    path = hierarchy_path(c["id"])
    path_str = " > ".join(path)                                   # 'Thing > Place > AdministrativeArea'
    head = f"{c['label']} ({path_str})" if path_str else c["label"]
    return f"{head}: {c['comment']}"

# Materialize the list once: class_list[i] and class_texts[i] line up by index,
# so after we embed class_texts we can map a row back to its class via class_list.
class_list = list(classes.values())
class_texts = [class_embedding_text(c) for c in class_list]

# sanity check: print the embedding text for a few familiar classes
for label in ["Country", "City", "NewsMediaOrganization", "GovernmentOrganization"]:
    hit = next((t for c, t in zip(class_list, class_texts) if c["label"] == label), None)
    print(hit, "\n")

Country (Thing > Place > AdministrativeArea): A country. 

City (Thing > Place > AdministrativeArea): A city or town. 

NewsMediaOrganization (Thing > Organization): A News/Media organization such as a newspaper or TV station. 

GovernmentOrganization (Thing > Organization): A governmental organization or agency. 



In [4]:
# Save the parsed schema.org classes (with hierarchy path + embedding text) to disk.
# This file doubles as a cache: later runs can load it directly instead of re-parsing the
# 1.5 MB JSON-LD, and you can eyeball it to check the parsing looks right.
# It is a shared schema asset, so it lives in SCHEMA_DIR (not the per-article folder).
schema_classes = [
    {
        "id": c["id"],                                   # 'schema:Country'
        "label": c["label"],                             # 'Country'
        "comment": c["comment"],                         # 'A country.'
        "path": " > ".join(hierarchy_path(c["id"])),     # 'Thing > Place > AdministrativeArea'
        "embedding_text": class_embedding_text(c),       # the full string we embed
    }
    for c in class_list
]

out = SCHEMA_DIR / "schema_classes.json"
out.write_text(json.dumps(schema_classes, indent=2, ensure_ascii=False))
print(f"wrote {len(schema_classes)} classes -> {out}")

wrote 933 classes -> schema_org_files/schema_classes.json


## 2. Load OIE entities and build per-type query strings

We match at the **OIE type** level: one query per distinct free-form type (`type: definition`), with the mention left out. Entities are kept so we can map each type's match back onto every entity that carries it.

In [5]:
# The entities file has up to two parts:
#   "entities": [{text, type, confidence}, ...]   e.g. {"text": "Lebanon", "type": "Country", ...}
#   "types":    [{type, definition}, ...]          one-line definition per free-form type (OPTIONAL)
# Some articles (e.g. world_cup_article) ship entities without a "types" block; we handle that.
oie = json.loads(OIE_ENTITIES.read_text())

# Build a lookup: free-form type label -> its definition. e.g.
#   "Country" -> "A sovereign nation-state."
# Empty when the file has no "types" block.
type_defs = {t["type"]: t["definition"] for t in oie.get("types", [])}
entities = oie["entities"]

# The matching unit is the DISTINCT OIE type, not the mention. Collect the distinct types in
# first-seen order; every entity will later inherit the match found for its type.
oie_types = list(dict.fromkeys(e["type"] for e in entities))

def type_query_text(t):
    """Query string we embed for one OIE type = type label + its definition (NO mention).

    Example (type "Country"): "Country: A sovereign nation-state."
    Dropping the mention is the whole point: the same type always embeds to the same vector, so
    every "Country" entity gets the same schema match and mention tokens can't perturb it.
    """
    definition = type_defs.get(t, "")
    return f"{t}: {definition}"

type_texts = [type_query_text(t) for t in oie_types]   # lines up with `oie_types` by index
print(f"{len(entities)} entities across {len(oie_types)} distinct OIE types "
      f"| {len(type_defs)} type definitions")
for t in type_texts[:5]:
    print(" ", t)

38 entities across 20 distinct OIE types | 20 type definitions
  Military Force: An organized armed body belonging to a state or political entity that conducts military operations.
  Region: A geographically defined sub-national area or zone within or spanning countries.
  Person: An individual human being identified by name.
  Country: A sovereign nation-state.
  Temporal Expression: A word or phrase referring to a specific point or period in time, such as a day or time of day.


## 3. Embed and retrieve top-k

In [6]:
import numpy as np
from sentence_transformers import SentenceTransformer

# Download (first time) + load the embedding model. It turns each string into a fixed-length
# vector; strings with similar meaning end up close together in that vector space.
model = SentenceTransformer(MODEL_NAME)

# normalize_embeddings=True scales every vector to length 1. That matters because then the
# dot product between two vectors EQUALS their cosine similarity (a number in [-1, 1] where
# higher = more similar) — which is exactly what we want and makes the next cell a single matmul.
#
# Documents (schema classes) are embedded as-is; queries (OIE types) get the BGE instruction prefix.
class_emb = model.encode(class_texts, normalize_embeddings=True,
                         show_progress_bar=True, batch_size=64)
type_emb = model.encode([QUERY_INSTRUCTION + t for t in type_texts],
                        normalize_embeddings=True, show_progress_bar=True, batch_size=64)

class_emb = np.asarray(class_emb)   # shape (n_classes, dim)   e.g. (933, 384)
type_emb = np.asarray(type_emb)     # shape (n_types,   dim)   e.g. (34, 384)
print("class_emb", class_emb.shape, "| type_emb", type_emb.shape)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/15 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

class_emb (933, 768) | type_emb (20, 768)


In [7]:
# One matrix multiply gives every OIE-type-vs-every-class similarity at once.
# type_emb is (n_types, 384), class_emb.T is (384, 933) -> type_sims is (n_types, 933).
type_sims = type_emb @ class_emb.T
type_topk = np.argsort(-type_sims, axis=1)[:, :TOP_K]   # best-first column indices per type

# top-K schema candidates for each DISTINCT OIE type
type_results = {}
for ti, t in enumerate(oie_types):
    type_results[t] = [
        {"label": class_list[j]["label"],
         "score": round(float(type_sims[ti, j]), 3),
         "path": " > ".join(hierarchy_path(class_list[j]["id"])),
         "comment": class_list[j]["comment"]}
        for j in type_topk[ti]
    ]

# Per-entity results: every entity inherits its OIE type's match (identical within a type).
results = [
    {"entity": e["text"], "free_type": e["type"], "candidates": type_results[e["type"]]}
    for e in entities
]

## 4. Inspect results

In [8]:
# Full view: each DISTINCT OIE type with its top-K schema candidates.
# Every entity of a type gets exactly this set, so we show it once per type (not per mention).
# Example line:  "  0.812  schema:Country               (Thing > Place > AdministrativeArea)"
for t in oie_types:
    print(f"\n▶ {t}")
    for c in type_results[t]:
        print(f"    {c['score']:.3f}  schema:{c['label']:<22} ({c['path']})")


▶ Military Force
    0.813  schema:DefenceEstablishment   (Thing > Place > CivicStructure > GovernmentBuilding)
    0.812  schema:LegalForceStatus       (Thing > Intangible > Enumeration > StatusEnumeration)
    0.802  schema:GovernmentService      (Thing > Intangible > Service)
    0.801  schema:State                  (Thing > Place > AdministrativeArea)
    0.797  schema:PoliticalParty         (Thing > Organization)

▶ Region
    0.867  schema:DefinedRegion          (Thing > Place)
    0.848  schema:AdministrativeArea     (Thing > Place)
    0.828  schema:State                  (Thing > Place > AdministrativeArea)
    0.824  schema:Continent              (Thing > Place > Landform)
    0.821  schema:Country                (Thing > Place > AdministrativeArea)

▶ Person
    0.872  schema:Person                 (Thing)
    0.830  schema:Residence              (Thing > Place)
    0.822  schema:Question               (Thing > CreativeWork > Comment)
    0.822  schema:IndividualProduct    

In [9]:
# Condensed view: the single best (top-1) schema class per DISTINCT OIE type.
#
# GAP_THRESHOLD is the cutoff for "the embedding didn't find a confident match". If even the
# BEST candidate scores below it, the type may not fit any existing schema.org class — i.e.
# a candidate for the extension/minting step. 0.55 is just a starting guess for bge-small.
GAP_THRESHOLD = 0.55

for t in oie_types:
    best = type_results[t][0]   # candidates are best-first, so [0] is the top match
    flag = "  <-- possible gap / mint candidate" if best["score"] < GAP_THRESHOLD else ""
    print(f"{best['score']:.3f}  {t:<28} -> schema:{best['label']}{flag}")

# A mention can still be typed more than once (e.g. "Mexico" as Country AND National football
# team). Those are different types, so each typing keeps its own schema match — shown here.
from collections import defaultdict
by_mention = defaultdict(dict)              # mention -> {free_type: matched schema label}
for r in results:
    by_mention[r["entity"]][r["free_type"]] = r["candidates"][0]["label"]
multi = {m: d for m, d in by_mention.items() if len(d) > 1}
if multi:
    print(f"\nMentions typed more than once ({len(multi)}) — matched per typing:")
    for m, d in multi.items():
        mapping = ", ".join(f"{ft} -> schema:{lbl}" for ft, lbl in d.items())
        print(f"  {m:<30} {mapping}")

0.813  Military Force               -> schema:DefenceEstablishment
0.867  Region                       -> schema:DefinedRegion
0.872  Person                       -> schema:Person
0.877  Country                      -> schema:Country
0.842  Temporal Expression          -> schema:Time
0.824  Military Aircraft            -> schema:Airline
0.836  Locality                     -> schema:Residence
0.845  News Agency                  -> schema:NewsMediaOrganization
0.818  Armed Group                  -> schema:NGO
0.811  Summit                       -> schema:Mountain
0.865  City                         -> schema:City
0.813  International Agreement      -> schema:AgreeAction
0.901  Resort                       -> schema:Resort
0.829  Government Ministry          -> schema:GovernmentOrganization
0.883  Newspaper                    -> schema:Newspaper
0.886  Body of Water                -> schema:SeaBodyOfWater
0.835  Document Collection          -> schema:Collection
0.843  Religious Site      

In [10]:
# Persist results (per-article outputs). We save two views of the same matching:
#   - per TYPE  : the actual matching unit (one entry per distinct OIE type)
#   - per ENTITY: every entity with its type's match (handy for downstream joins)
out_types = ARTICLE_DIR / "schema_matching_trial_results_by_type.json"
out_types.write_text(json.dumps(type_results, indent=2, ensure_ascii=False))

out = ARTICLE_DIR / "schema_matching_trial_results.json"
out.write_text(json.dumps(results, indent=2, ensure_ascii=False))
print("wrote", out_types, "and", out)

wrote politics_article/schema_matching_trial_results_by_type.json and politics_article/schema_matching_trial_results.json


## 5. Variant: LLM-enriched schema definitions

Replace each schema.org type's terse comment (e.g. `"A country."`) with a richer Claude-authored definition — expanded gloss + synonyms/aka + a couple of example instances — pre-built into `schema_org_files/schema_classes_enriched.json` by [build_enriched_schema.py](schema_org_files/build_enriched_schema.py). (Contrastive "distinct from …" cues are kept in `enriched_comment` for a later LLM judge but stripped from the embedded text, since a bi-encoder reads them as similarity, not contrast.)

**All 933 types are enriched** (a one-time, reusable artifact — the same file works for every article). The query side is unchanged and `type_emb` is reused, so any difference comes purely from the enriched document text.

We compare against the baseline `type_results` and watch two things: top-1 changes, and the **top1–top2 margin** — a wider margin means the top pick pulled decisively ahead of its runner-up, which is a more meaningful "confident match?" signal than the raw cosine score (whose absolute value just shifts when you change the text).

In [11]:
# Load the LLM-enriched schema pool (built once by schema_org_files/build_enriched_schema.py)
# and embed it. Each row has the SAME shape as schema_classes.json plus: enriched_comment,
# enriched (bool). `embedding_text` uses the enriched comment in the same "Label (path): comment"
# format as the baseline -> isolates the effect of enrichment.
enriched_classes = json.loads((SCHEMA_DIR / "schema_classes_enriched.json").read_text())
print(f"{len(enriched_classes)} classes, "
      f"{sum(c['enriched'] for c in enriched_classes)} with enriched definitions")

enr_texts = [c["embedding_text"] for c in enriched_classes]
enr_emb = np.asarray(model.encode(enr_texts, normalize_embeddings=True,
                                  show_progress_bar=True, batch_size=64))

# Reuse the SAME per-type query embeddings (type_emb); only the documents differ.
type_sims_enr = type_emb @ enr_emb.T
type_topk_enr = np.argsort(-type_sims_enr, axis=1)[:, :TOP_K]

# top-K enriched candidates per DISTINCT OIE type
type_results_enriched = {}
for ti, t in enumerate(oie_types):
    type_results_enriched[t] = [
        {"label": enriched_classes[j]["label"],
         "score": round(float(type_sims_enr[ti, j]), 3),
         "path": enriched_classes[j]["path"],
         "enriched": enriched_classes[j]["enriched"],
         "comment": enriched_classes[j]["enriched_comment"] or enriched_classes[j]["comment"]}
        for j in type_topk_enr[ti]
    ]

# Per-entity view (every entity inherits its type's enriched match).
results_enriched = [
    {"entity": e["text"], "free_type": e["type"], "candidates": type_results_enriched[e["type"]]}
    for e in entities
]
print("type_results_enriched ready")

933 classes, 933 with enriched definitions


Batches:   0%|          | 0/15 [00:00<?, ?it/s]

type_results_enriched ready


In [12]:
# --- Compare baseline schema comments vs LLM-enriched definitions (per OIE type) ------------
#   OLD = original schema.org comment   (type_results)
#   NEW = Claude-enriched definition    (type_results_enriched)
# Same query side; only the schema document text changed (across the whole 933-type pool).
# We track top-1 changes AND the top1-top2 margin: a bigger margin = a more decisive match,
# which is a better "is this confident?" signal than the raw cosine score.
import statistics

def margin(cands):
    return cands[0]["score"] - cands[1]["score"] if len(cands) > 1 else float("nan")
def fmt(cands):
    return "  ".join(f"{c['label']}({c['score']:.2f})" for c in cands)

flips, dmargin = [], []
for t in oie_types:
    old, new = type_results[t], type_results_enriched[t]
    same = old[0]["label"] == new[0]["label"]
    mo, mn = margin(old), margin(new)
    dmargin.append(mn - mo)
    if not same:
        flips.append((t, old[0]["label"], new[0]["label"]))
    mark = "=" if same else "x"
    print(f"\n[{mark}] {t}")
    print(f"    OLD comment : {fmt(old)}    margin {mo:+.3f}")
    print(f"    NEW enriched: {fmt(new)}    margin {mn:+.3f}")

n = len(oie_types)
mean_old = statistics.mean(margin(type_results[t]) for t in oie_types)
mean_new = statistics.mean(margin(type_results_enriched[t]) for t in oie_types)
print(f"\ntop-1 agreement: {n - len(flips)}/{n} = {(n - len(flips))/n:.0%}")
print(f"mean top1-top2 margin  OLD -> NEW: {mean_old:.3f} -> {mean_new:.3f}")
print(f"mean margin change (new-old): {statistics.mean(dmargin):+.3f}  "
      f"(positive = enriched defs make the top pick more decisive)")
if flips:
    print("\ntop-1 changes (type: OLD -> NEW):")
    for t, o, nw in flips:
        print(f"  {t:<28} {o} -> {nw}")


[=] Military Force
    OLD comment : DefenceEstablishment(0.81)  LegalForceStatus(0.81)  GovernmentService(0.80)  State(0.80)  PoliticalParty(0.80)    margin +0.001
    NEW enriched: DefenceEstablishment(0.83)  PoliticalParty(0.82)  LegalForceStatus(0.82)  GovernmentService(0.80)  AnatomicalSystem(0.80)    margin +0.010

[=] Region
    OLD comment : DefinedRegion(0.87)  AdministrativeArea(0.85)  State(0.83)  Continent(0.82)  Country(0.82)    margin +0.019
    NEW enriched: DefinedRegion(0.88)  State(0.83)  AdministrativeArea(0.82)  Continent(0.82)  Country(0.81)    margin +0.049

[=] Person
    OLD comment : Person(0.87)  Residence(0.83)  Question(0.82)  IndividualProduct(0.82)  GenderType(0.82)    margin +0.042
    NEW enriched: Person(0.88)  AboutPage(0.84)  ProfilePage(0.83)  SearchAction(0.82)  ContactPage(0.82)    margin +0.047

[=] Country
    OLD comment : Country(0.88)  State(0.85)  Embassy(0.80)  URL(0.80)  SearchResultsPage(0.80)    margin +0.024
    NEW enriched: Country(0.

In [13]:
# Persist the enriched-definition results (per-article outputs), per type and per entity.
out_types_enr = ARTICLE_DIR / "schema_matching_trial_results_enriched_by_type.json"
out_types_enr.write_text(json.dumps(type_results_enriched, indent=2, ensure_ascii=False))

out_enr = ARTICLE_DIR / "schema_matching_trial_results_enriched.json"
out_enr.write_text(json.dumps(results_enriched, indent=2, ensure_ascii=False))
print("wrote", out_types_enr, "and", out_enr)

wrote politics_article/schema_matching_trial_results_enriched_by_type.json and politics_article/schema_matching_trial_results_enriched.json


In [14]:
# Summary: each OIE type with a few example mentions and its top-5 schema.org matches
# (from the LLM-enriched run). One-glance view of how each free-form type was canonicalized.
from collections import defaultdict

mentions_by_type = defaultdict(list)
for e in entities:
    if e["text"] not in mentions_by_type[e["type"]]:      # dedup, keep first-seen order
        mentions_by_type[e["type"]].append(e["text"])

N_EXAMPLES = 5
for t in oie_types:
    mentions = mentions_by_type[t]
    examples = ", ".join(mentions[:N_EXAMPLES])
    more = "" if len(mentions) <= N_EXAMPLES else f" (+{len(mentions) - N_EXAMPLES} more)"
    top5 = "  ".join(f"{c['label']}({c['score']:.2f})" for c in type_results_enriched[t])
    print(f"\n▶ {t}  [{len(mentions)} mention(s)]")
    print(f"   e.g.: {examples}{more}")
    print(f"   top-5: {top5}")


▶ Military Force  [2 mention(s)]
   e.g.: Israeli forces, The Israeli military
   top-5: DefenceEstablishment(0.83)  PoliticalParty(0.82)  LegalForceStatus(0.82)  GovernmentService(0.80)  AnatomicalSystem(0.80)

▶ Region  [1 mention(s)]
   e.g.: southern Lebanon
   top-5: DefinedRegion(0.88)  State(0.83)  AdministrativeArea(0.82)  Continent(0.82)  Country(0.81)

▶ Person  [4 mention(s)]
   e.g.: Donald Trump, Benjamin Netanyahu, Abbas Araghchi, Barack Obama
   top-5: Person(0.88)  AboutPage(0.84)  ProfilePage(0.83)  SearchAction(0.82)  ContactPage(0.82)

▶ Country  [8 mention(s)]
   e.g.: Israel, Lebanon, US, Iran, Pakistan (+3 more)
   top-5: Country(0.87)  State(0.83)  DrugStrength(0.80)  AdministrativeArea(0.79)  SearchResultsPage(0.79)

▶ Temporal Expression  [5 mention(s)]
   e.g.: Wednesday, Tuesday, Sunday night, Monday, Friday
   top-5: Time(0.85)  DateTime(0.84)  Duration(0.82)  SeekToAction(0.82)  Event(0.81)

▶ Military Aircraft  [1 mention(s)]
   e.g.: Israeli jets
   top-